# Tools Integration

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_comminity.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

In [ ]:
load_dotenv() # to use openai

In [ ]:
llm = ChatOpenAI()

In [ ]:
# Tools
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "subtract":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                raise {"error": "Division by zero is not allowed."}
            result = first_num / second_num
        else:
            raise {"error": f"Unsupported operation: {operation}"}
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        raise {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol )e.g. - 'AAPL', 'TSLA')
    using Alpha Vantage API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=demo" # put my own API key instead of `demo` without quotes
    response = requests.get(url) 
    return response.json()  # returns the JSON response from the API

In [ ]:
# Make tool list
tools = [get_stock_price, calculator, search_tool]

# make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessages], add_messages]

In [ ]:
# graph nodes
def chat_node(state: ChatState):
    messages = state["messages"]

    # send to model
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools) # Executes tool calls

In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

In [ ]:
graph.add_edge(START, 'chat_node')

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges('chat_node', tools_condition)

chatbot = graph.compile()
chatbot

In [ ]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple?")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the price of 50 shares apple?")]})
print(out["messages"][-1].content)

In [ ]:
# change the graph structure
graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges('chat_node', tools_condition)

graph.add_edge("tools", "chat_node")

chatbot = graph.compile()
chatbot

In [ ]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple?")]})
print(out["messages"][-1].content)

In [ ]:
# chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the price of 50 shares apple?")]})
print(out["messages"][-1].content)